In [45]:
!pip install -q torch transformers datasets tqdm peft
%pip uninstall -q -y torchao

Note: you may need to restart the kernel to use updated packages.


# 1. libs

In [46]:
import torch
from torch.utils.data import DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.optim import AdamW
from tqdm.auto import tqdm
from contextlib import nullcontext
from torch.optim import AdamW
from peft import LoraConfig, TaskType, get_peft_model
import torch.nn.functional as F
import re
from tqdm.auto import tqdm
import numpy as np

In [47]:
import gc
import torch

def gpu_memory_report():
    if not torch.cuda.is_available():
        print("CUDA is unavailable.")
        return

    free, total = torch.cuda.mem_get_info()
    GiB = 1024 ** 3

    print(f"GPU total:       {total / GiB:.2f} GiB")
    print(f"GPU free:        {free / GiB:.2f} GiB")
    print(f"PyTorch allocated: {torch.cuda.memory_allocated() / GiB:.2f} GiB")
    print(f"PyTorch reserved:  {torch.cuda.memory_reserved() / GiB:.2f} GiB")

def clean_unused_gpu_memory():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    gpu_memory_report()

clean_unused_gpu_memory()

GPU total:       14.56 GiB
GPU free:        9.52 GiB
PyTorch allocated: 3.31 GiB
PyTorch reserved:  4.91 GiB


# 2. cfg 

In [48]:
max_length = 384
max_input_length = 128
batch_size = 6
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", device)
MODEL_NAME = "Qwen/Qwen2.5-0.5B"


Device: cuda:0


# 3. data

In [49]:


dataset = load_dataset("openai/gsm8k", "main")

train_val = dataset["train"].train_test_split(
    test_size=256,
    seed=42
)

train_dataset = train_val["train"]
val_dataset = train_val["test"]
test_dataset = dataset["test"]


def collate_fn(batch, tokenizer, max_length, max_input_length):
    input_ids_list = []
    labels_list = []

    for ex in batch:
        prompt = f"Question: {ex['question']}\nAnswer:"
        answer = f"{ex['answer']}{tokenizer.eos_token}"
        prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"][-max_input_length:]
        answer_ids = tokenizer(answer, add_special_tokens=False)["input_ids"][:max_length - len(prompt_ids)]
        input_ids = prompt_ids + answer_ids
        labels = [-100] * len(prompt_ids) + answer_ids
        input_ids_list.append(input_ids)
        labels_list.append(labels)

    batch_max_len = max(len(ids) for ids in input_ids_list)
    padded_input_ids, padded_labels, attention_masks = [], [], []

    for input_ids, labels in zip(input_ids_list, labels_list):
        pad_len = batch_max_len - len(input_ids)
        padded_input_ids.append(input_ids + [tokenizer.pad_token_id] * pad_len)
        padded_labels.append(labels + [-100] * pad_len)
        attention_masks.append([1] * len(input_ids) + [0] * pad_len)

    return {
        "input_ids": torch.tensor(padded_input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(padded_labels, dtype=torch.long),
    }
    
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=lambda batch: collate_fn(
        batch = batch,
        tokenizer=tokenizer,
        max_length=max_length,
        max_input_length=max_input_length
    )
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=lambda batch: collate_fn(
        batch = batch,
        tokenizer=tokenizer,
        max_length=max_length,
        max_input_length=max_input_length
    )
)


print(len(train_dataset))  
print(len(val_dataset))  
print(len(test_dataset)) 

print(type(train_dataset))
train_dataset[4]

7217
256
1319
<class 'datasets.arrow_dataset.Dataset'>


{'question': 'Laura bought 2 pairs of pants for $54 each and 4 shirts for $33 each. She gave $250 to the cashier. So how much change did she take?',
 'answer': '54*2=$<<54*2=108>>108 is the price of 2 pairs of pants.\n4*33=$<<4*33=132>>132 is the price of 4 shirts.\n108+132=$<<108+132=240>>240 is the total price.\nSo the cashier gave 250-240=$<<250-240=10>>10 change to Laura.\n#### 10'}

In [50]:
# toks_q = []
# toks_a = []

# for row in train_dataset:
#     q = row['question']
#     a = row['answer']
#     tok_q = tokenizer(q, add_special_tokens=False)
#     tok_a = tokenizer(a, add_special_tokens=False)
#     toks_q.append(tok_q)
#     toks_a.append(tok_a)



In [51]:
# len_tok_q = [len(c['input_ids']) for c in toks_q]
# len_tok_a = [len(c['input_ids']) for c in toks_a]

# print(np.percentile(len_tok_q, q = [10,50, 80, 95, 99, 99.5, 100]))
# print(np.percentile(len_tok_a, q = [10,50, 80, 95, 99, 99.5, 100]))


# Model 

In [52]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Special Tokens:", tokenizer.all_special_tokens)


model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float32,
    attn_implementation="sdpa",
)


lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.to(device)
model.print_trainable_parameters()

Special Tokens: ['<|endoftext|>', '<|im_start|>', '<|im_end|>', '<|object_ref_start|>', '<|object_ref_end|>', '<|box_start|>', '<|box_end|>', '<|quad_start|>', '<|quad_end|>', '<|vision_start|>', '<|vision_end|>', '<|vision_pad|>', '<|image_pad|>', '<|video_pad|>']


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


# train helper

In [58]:

def calc_loss(logits, labels):
    loss = F.cross_entropy(
        logits.view(-1, logits.size(-1)),
        labels.view(-1),
        ignore_index=-100,
    )
    return loss

def train_one_epoch(train_loader, model):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    num_batches = len(train_loader)
    accum_steps = 4   # effective batch size = batch_size * accum_steps

    with tqdm(enumerate(train_loader), total=num_batches) as pbar:
        for batch_idx, batch in pbar:
            x_in = batch['input_ids'].to(device)
            y_label = batch['labels'].to(device)
            attention_mask = batch['attention_mask'].to(device)

            outputs = model(input_ids=x_in, attention_mask=attention_mask)
            logits = outputs.logits
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = y_label[:, 1:].contiguous()
            loss = calc_loss(shift_logits, shift_labels)

            (loss / accum_steps).backward() 

            is_update_step = (batch_idx + 1) % accum_steps == 0
            is_last_batch = (batch_idx + 1) == num_batches
            if is_update_step or is_last_batch:
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)

            pbar.set_postfix({'train_loss': loss.item()})  
            

def eval_val_set(model, val_loader):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        with tqdm(enumerate(val_loader), total=len(val_loader)) as pbar:
            for batch_idx, batch in pbar:
                x_in = batch['input_ids'].to(device)
                y_label = batch['labels'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                outputs = model(input_ids=x_in, attention_mask=attention_mask)
                logits = outputs.logits
                shift_logits = logits[:, :-1, :].contiguous()
                shift_labels = y_label[:, 1:].contiguous()
                loss = calc_loss(shift_logits, shift_labels)
                total_loss += loss.item()
                pbar.set_postfix({'val_loss': total_loss / (batch_idx + 1)})
    return total_loss / len(val_loader)
    
def generate_seq(model, list_of_questions, max_seq_len=512):
    input_seq_ids = []
    for question in list_of_questions:
        prompt = f"Question: {question}\nAnswer:"
        prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        input_seq_ids.append(prompt_ids)

    
    max_input_seq = max(len(x) for x in input_seq_ids)
    padded_input_ids = []
    attention_masks = []
    for input_ids in input_seq_ids:
        left_side_pad_len = max_input_seq - len(input_ids)
        padded_input_ids.append([tokenizer.pad_token_id] * left_side_pad_len + input_ids)
        attention_masks.append([0] * left_side_pad_len + [1] * len(input_ids))

    
    x_in = torch.tensor(padded_input_ids, dtype=torch.long, device=device)
    attn_mask = torch.tensor(attention_masks, dtype=torch.long, device=device)
    
    model.eval()
    finished = torch.zeros(x_in.size(0), dtype=torch.bool, device=device)
    for _ in range(max_seq_len):
        with torch.no_grad():
            outputs = model(input_ids=x_in, attention_mask=attn_mask)

        logits = outputs.logits[:, -1, :]
        preds_token_id = logits.argmax(dim=-1)
        preds_token_id = torch.where(finished, tokenizer.eos_token_id, preds_token_id)
        
        x_in = torch.cat([x_in, preds_token_id.unsqueeze(1)], dim=1)
        attn_mask = torch.cat([attn_mask, (~finished).long().unsqueeze(1)], dim=1)
        finished = finished | preds_token_id.eq(tokenizer.eos_token_id)
        if finished.all():
            break
    return [tokenizer.decode(seq[max_input_seq:], add_special_tokens=False) for seq in x_in]



def extract_answer(text):
    matches = re.findall(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)", text)
    return matches[-1].replace(",", "") if matches else None

def evaluate_gsm8k(model, test_dataset, batch_size=8, max_new_tokens=256):
    model.eval()
    correct = 0
    total = 0
    predictions = []
    for start in tqdm(range(0, len(test_dataset), batch_size)):
        batch = test_dataset.select(range(start, min(start + batch_size, len(test_dataset))))
        questions = batch["question"]
        references = [extract_answer(a) for a in batch["answer"]]
        generated = generate_seq(model, questions, max_seq_len=max_new_tokens)
        for question, prediction, reference in zip(questions, generated, references):
            predicted_answer = extract_answer(prediction)
            is_correct = predicted_answer is not None and reference is not None and predicted_answer == reference
            correct += int(is_correct)
            total += 1
            predictions.append({
                "question": question,
                "prediction": prediction,
                "predicted_answer": predicted_answer,
                "reference": reference,
                "correct": is_correct
            })
    accuracy = correct / total if total else 0.0
    print(f"Test accuracy: {accuracy:.2%} ({correct}/{total})")
    return accuracy, predictions



In [59]:
test_dataset[0:4]

{'question': ["Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?",
  'A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?',
  'Josh decides to try flipping a house.  He buys a house for $80,000 and then puts in $50,000 in repairs.  This increased the value of the house by 150%.  How much profit did he make?',
  'James decides to run 3 sprints 3 times a week.  He runs 60 meters each sprint.  How many total meters does he run a week?'],
 'answer': ['Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18',
  'It takes 2/2=<<2/2=1>>1 bolt of white fiber\nSo the total amount of fabric is 2+1=<<2+1=3>>3 bolts of fabric\n#### 3',
  'The 

In [60]:
optimizer = AdamW(model.parameters(), lr=3e-5)
model.to(device)
model.config.use_cache = False
model.gradient_checkpointing_enable()
# train_one_epoch(val_loader, model)

In [61]:
NUM_EPOCHS = 5
clean_unused_gpu_memory()

for epoch in range(NUM_EPOCHS):
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS}")
    train_one_epoch(train_loader, model)
    val_loss = eval_val_set(model, val_loader)
    
    print(f"Validation Loss: {val_loss:.4f}")

    
    list_of_questions = test_dataset[:5]["question"]
    outputs = generate_seq(model, list_of_questions, max_seq_len=20)
    
    for i, (question, output, answer) in enumerate(zip(list_of_questions, outputs, test_dataset[:5]["answer"]), 1):
        print(f"\nExample {i}")
        print(f"Question: {question}")
        print(f"Generated: {output}")
        print(f"Expected: {answer}")
        print("-" * 80)
    clean_unused_gpu_memory()


GPU total:       14.56 GiB
GPU free:        8.62 GiB
PyTorch allocated: 3.25 GiB
PyTorch reserved:  5.81 GiB
Epoch 1/5


  0%|          | 0/1203 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.30 GiB. GPU 0 has a total capacity of 14.56 GiB of which 874.81 MiB is free. Including non-PyTorch memory, this process has 13.71 GiB memory in use. Of the allocated memory 8.65 GiB is allocated by PyTorch, and 4.92 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [57]:
test_accuracy, test_predictions = evaluate_gsm8k(model, test_dataset)
print(f"Test Accuracy: {test_accuracy:.2%}")

  0%|          | 0/165 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
for x in range(5):
    train_one_epoch(val_loader, model)

In [ ]:


for question, answer in zip(samp_ds['question'], samp_ds['answer']):

    input_text = f"Question: {question} Answer:"
    output_text = f"{answer} {eos_token}"
    
    prompt_tokens_id = tokenizer(input_text, add_special_tokens=False)["input_ids"]
    output_tokens_id = tokenizer(output_text, add_special_tokens=False)["input_ids"]
    
    input_tokens_id = prompt_tokens_id + output_tokens_id
    labels = [-100]*len(prompt_tokens_id) + output_tokens_id
    
    x_in = torch.tensor([input_tokens_id]).to(device)
    y_label = torch.tensor([labels]).to(device)

    
    outputs = model(input_ids = x_in)
    logits = outputs.logits
    
    shift_logits = logits[:, :-1, :].contiguous()
    shift_labels = y_label[:, 1:].contiguous()
    
    manual_loss = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)),
        shift_labels.view(-1),
        ignore_index=-100,
    )
    
    print("Manual loss:", manual_loss.item())
    
    optimizer.zero_grad()
    manual_loss.backward()
    optimizer.step()

In [ ]:
import torch.nn.functional as F

logits = outputs.logits

shift_logits = logits[:, :-1, :].contiguous()
shift_labels = y_label[:, 1:].contiguous()

manual_loss = F.cross_entropy(
    shift_logits.view(-1, shift_logits.size(-1)),
    shift_labels.view(-1),
    ignore_index=-100,
)

print("Manual loss:", manual_loss.item())

In [ ]:
shift_logits.shape, shift_labels.shape, logits[:, :-1, :].contiguous().shape, logits.shape

In [ ]:
from torch.optim import AdamW
optimizer = AdamW(model.parameters(), lr=2e-5)
optimizer.zero_grad()
manual_loss.backward()
optimizer.step()

In [ ]:
prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
answer_ids = tokenizer(answer, add_special_tokens=False)["input_ids"]

print(prompt_ids[:10])
print(answer_ids[:10])

print("Prompt length:", len(prompt_ids))
print("Answer length:", len(answer_ids))

print(tokenizer.decode(prompt_ids[:10]))
print(tokenizer.decode(answer_ids[:10]))

In [ ]:
input_ids = prompt_ids + answer_ids + [tokenizer.eos_token_id]
labels = [-100] * len(prompt_ids) + answer_ids + [tokenizer.eos_token_id]

In [ ]:
len(input_ids), len(labels)

In [ ]:
input_ids_tokens = torch.tensor([input_ids]).to(model.device)
labels = torch.tensor([labels]).to(model.device)

outputs = model(
    input_ids=input_ids_tokens,
    labels=labels
)

print("Loss:", outputs.loss.item())
print("Logits shape:", outputs.logits.shape)

In [ ]:
loss = outputs.loss

In [ ]:
import torch.nn.functional as F

logits = outputs.logits

shift_logits = logits[:, :-1, :].contiguous()
shift_labels = labels[:, 1:].contiguous()

manual_loss = F.cross_entropy(
    shift_logits.view(-1, shift_logits.size(-1)),
    shift_labels.view(-1),
    ignore_index=-100,
)

print("Model loss:", outputs.loss.item())
print("Manual loss:", manual_loss.item())

In [ ]:
outputs.logits.shape

In [ ]:
outputs.past_key_values

In [ ]:
shift_logits = logits[:, :-1, :].contiguous()
shift_labels = labels[:, 1:].contiguous()

shift_logits.shape, shift_labels.shape

In [ ]:
final_probs = F.softmax(shift_logits[0][-1])
final_probs.shape

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Create a 1D numpy array
y_values = final_probs.detach().numpy()

# Plot the array
plt.plot(y_values, marker='o', linestyle='-', color='b')

# Add labels and a title
plt.title("Line Plot of a 1D Array")
plt.xlabel("Index")
plt.ylabel("Value")

# Display the plot
plt.show()


In [ ]:
word2idx = tokenizer.get_vocab() 
idx2word = {val: key for key, val in word2idx.items()}
idx2word.get(0)

In [ ]:
idx2word[int(final_probs.argmax())]

In [ ]:
for name, param in model.named_parameters():
     print(name)
        